# Caso 03 — NidoHogar, portal inmobiliario: del clic al contrato
## Día 1 — Exploración Inicial y Auditoría Forense de Datos (EDA)

**Contexto del encargo:**
**Álvaro Nadal (CEO)** debe decidir el viernes si aprueba el lanzamiento de **«Destacado Premium»**, la propuesta de **Gonzalo Prieto (Director Comercial)** para vender aún más visibilidad a las agencias inmobiliarias bajo el argumento de que *«los anuncios destacados reciben casi el doble de visitas y contactos que los normales»*. **Cristina Márquez (PM del Marketplace)** sospecha que muchos contactos caen en saco roto y **Irene Salas (Lead de Datos)** nos pide auditar primero la base antes de cruzar tablas.

**Objetivo del Día 1:**
Realizar la radiografía forense de las **7 tablas en `data/raw/`** sin modificar los ficheros originales, respondiendo a las 4 preguntas de rigor en cada tabla:
1. **Tamaño (`.shape`):** filas y columnas reales frente a identificadores únicos.
2. **Pinta (`.head()`):** inspección visual de formatos de moneda, texto y fechas.
3. **Tipos (`.dtypes`):** detección de columnas numéricas o temporales almacenadas como texto (`object`/`str`).
4. **Huecos (`.isnull().sum()`):** distinción entre nulos técnicos e **indicadores de negocio ocultos tras un vacío** (`fecha_baja`, `fecha_respuesta`, `plan_suscripcion`, `comision`).

In [1]:
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
RAW_DIR = os.path.join('data', 'raw')

archivos = [
    ('anuncios', 'anuncio_id', 'Dimensión central: inmuebles publicados'),
    ('agencias', 'agencia_id', 'Dimensión: agencias y particulares anunciantes'),
    ('zonas', 'zona', 'Referencia: barrios canónicos y precio medio/m²'),
    ('productos_visibilidad', 'pv_id', 'Hechos: compras de destacados y subidas'),
    ('visitas', 'visita_id', 'Hechos (arriba del embudo): clics a fichas'),
    ('contactos', 'contacto_id', 'Hechos (centro del embudo): mensajes/leads'),
    ('contratos', 'contrato_id', 'Hechos (fondo del embudo): alquileres cerrados'),
]

tablas_raw = {}
resumen_mapa = []
for nombre, pk, rol in archivos:
    df = pd.read_csv(os.path.join(RAW_DIR, f'{nombre}.csv'))
    tablas_raw[nombre] = df
    resumen_mapa.append({
        'tabla': f'{nombre}.csv',
        'rol_modelo': rol,
        'filas': df.shape[0],
        'columnas': df.shape[1],
        'pk_unicos': df[pk].nunique(),
        'duplicados_pk': df.shape[0] - df[pk].nunique(),
        'cols_con_nulos': int((df.isnull().sum() > 0).sum())
    })

df_mapa = pd.DataFrame(resumen_mapa)
print("=== MAPA GENERAL DE LAS 7 TABLAS CRUDAS DE NIDOHOGAR ===")
display(df_mapa)

=== MAPA GENERAL DE LAS 7 TABLAS CRUDAS DE NIDOHOGAR ===


,tabla,rol_modelo,filas,columnas,pk_unicos,duplicados_pk,cols_con_nulos
0,anuncios.csv,Dimensión central: inmuebles publicados,4700,12,4700,0,3
1,agencias.csv,Dimensión: agencias y particulares anunciantes,120,6,120,0,1
2,zonas.csv,Referencia: barrios canónicos y precio medio/m²,40,3,40,0,0
3,productos_visibilidad.csv,Hechos: compras de destacados y subidas,1697,6,1697,0,1
4,visitas.csv,Hechos (arriba del embudo): clics a fichas,55661,4,55275,386,0
5,contactos.csv,Hechos (centro del embudo): mensajes/leads,18005,6,17880,125,1
6,contratos.csv,Hechos (fondo del embudo): alquileres cerrados,1169,6,1169,0,1


### Análisis del Mapa General de Tablas Crudas
* El modelo relacional sigue la secuencia exacta del negocio (**Del clic al contrato**): **`4.700` anuncios** publicados por **`120` agencias/anunciantes** en **`40` zonas canónicas**, que reciben **`55.661` registros de visitas**, generan **`18.005` registros de contactos** y culminan en **`1.169` contratos firmados** (además de **`1.697` compras de visibilidad**).
* **Primera alerta forense (duplicados técnicos):** Las tablas de dimensiones (`anuncios`, `agencias`, `zonas`) y el cierre (`contratos`) tienen clave primaria íntegra (`0` duplicados), pero las dos tablas de eventos intermedios presentan reintentos de registro: **`386` filas duplicadas en `visitas.csv`** (`55.275` únicas) y **`125` filas duplicadas en `contactos.csv`** (`17.880` únicos).

In [2]:
for nombre in ['anuncios', 'agencias', 'zonas', 'productos_visibilidad']:
    df = tablas_raw[nombre]
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    print(f"=== {nombre}.csv | shape={df.shape} ===")
    print("Tipos:", df.dtypes.to_dict())
    print("Nulos:", nulos.to_dict() if len(nulos) > 0 else "Sin nulos")
    display(df.head(3))

=== anuncios.csv | shape=(4700, 12) ===
Tipos: {'anuncio_id': dtype('int64'), 'agencia_id': dtype('int64'), 'zona': <StringDtype(storage='python', na_value=nan)>, 'ciudad': <StringDtype(storage='python', na_value=nan)>, 'tipo_anuncio': <StringDtype(storage='python', na_value=nan)>, 'precio_mensual': <StringDtype(storage='python', na_value=nan)>, 'habitaciones': dtype('float64'), 'm2': <StringDtype(storage='python', na_value=nan)>, 'num_fotos': dtype('float64'), 'fecha_publicacion': <StringDtype(storage='python', na_value=nan)>, 'fecha_baja': <StringDtype(storage='python', na_value=nan)>, 'tipo_anunciante': <StringDtype(storage='python', na_value=nan)>}
Nulos: {'habitaciones': 141, 'num_fotos': 94, 'fecha_baja': 3798}


,anuncio_id,agencia_id,zona,ciudad,tipo_anuncio,precio_mensual,habitaciones,m2,num_fotos,fecha_publicacion,fecha_baja,tipo_anunciante
0,1,90,Jesús,VALENCIA,normal,"940,00 €",3.0,79,6.0,21/02/2024,NaN,agencia
1,2,45,Vallecas,Madrid,Normal,"310,00 €",0.0,25,12.0,16/12/2023,NaN,agencia
2,3,29,Churriana,MALAGA,Destacado,"850,00 €",2.0,67,13.0,13/12/2023,NaN,Particular


=== agencias.csv | shape=(120, 6) ===
Tipos: {'agencia_id': dtype('int64'), 'nombre': <StringDtype(storage='python', na_value=nan)>, 'ciudad': <StringDtype(storage='python', na_value=nan)>, 'tipo': <StringDtype(storage='python', na_value=nan)>, 'plan_suscripcion': <StringDtype(storage='python', na_value=nan)>, 'fecha_alta': <StringDtype(storage='python', na_value=nan)>}
Nulos: {'plan_suscripcion': 25}


,agencia_id,nombre,ciudad,tipo,plan_suscripcion,fecha_alta
0,1,Inmobiliaria La Marina,Sevilla,agencia,premium,29 Jan 2021
1,2,Hogar Génova,Valencia,Agencia,basico,10/09/2019
2,3,Pisos El Faro,VLC,agencia,Basico,28/03/2023


=== zonas.csv | shape=(40, 3) ===
Tipos: {'zona': <StringDtype(storage='python', na_value=nan)>, 'ciudad': <StringDtype(storage='python', na_value=nan)>, 'precio_medio_m2': <StringDtype(storage='python', na_value=nan)>}
Nulos: Sin nulos


,zona,ciudad,precio_medio_m2
0,Ruzafa,Valencia,"13,50 €"
1,El Carmen,Valencia,"12,80 €"
2,Benimaclet,Valencia,"10,20 €"


=== productos_visibilidad.csv | shape=(1697, 6) ===
Tipos: {'pv_id': dtype('int64'), 'anuncio_id': dtype('int64'), 'producto': <StringDtype(storage='python', na_value=nan)>, 'fecha_inicio': <StringDtype(storage='python', na_value=nan)>, 'fecha_fin': <StringDtype(storage='python', na_value=nan)>, 'precio': <StringDtype(storage='python', na_value=nan)>}
Nulos: {'fecha_fin': 638}


,pv_id,anuncio_id,producto,fecha_inicio,fecha_fin,precio
0,400000194,708,destacado,12/05/2024,16 Jun 2024,"29,90 €"
1,400000918,3133,destacado,22/04/2024,NaN,"39,90 €"
2,400001451,1368,subir_ranking,13/04/2024,2024-05-28,"14,90 €"


### Análisis de las Tablas de Dimensión y Catálogo (`anuncios`, `agencias`, `zonas`, `productos_visibilidad`)
1. **`anuncios.csv` (`4.700 × 12`):**
   * **Tipos sucios:** `precio_mensual` (`"940,00 €"`) y `m2` están guardados como texto, igual que `fecha_publicacion` y `fecha_baja`.
   * **Nulos con significado:** `fecha_baja` tiene **`3.798` nulos (`80,81 %`)**, que identifican a los **anuncios actualmente activos** en el portal (borrarlos destruiría el inventario). Además hay `94` nulos en `num_fotos` (anuncios publicados sin fotografía, imputables a `0`) y `141` en `habitaciones`.
2. **`agencias.csv` (`120 × 6`):**
   * `plan_suscripcion` presenta **`25` nulos**, correspondientes a anunciantes de tipo `particular` que publican sin cuota mensual de agencia.
3. **`zonas.csv` (`40 × 3`) y `productos_visibilidad.csv` (`1.697 × 6`):**
   * Tanto `precio_medio_m2` (`"13,50 €"`) como `precio` (`"29,90 €"`) vienen en formato moneda europea como texto; `fecha_fin` tiene `638` nulos en promociones aún vigentes o puntuales.

In [3]:
for nombre in ['visitas', 'contactos', 'contratos']:
    df = tablas_raw[nombre]
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    print(f"=== {nombre}.csv | shape={df.shape} ===")
    print("Tipos:", df.dtypes.to_dict())
    print("Nulos:", nulos.to_dict() if len(nulos) > 0 else "Sin nulos")
    display(df.head(3))

=== visitas.csv | shape=(55661, 4) ===
Tipos: {'visita_id': dtype('int64'), 'anuncio_id': dtype('int64'), 'fecha': <StringDtype(storage='python', na_value=nan)>, 'dispositivo': <StringDtype(storage='python', na_value=nan)>}
Nulos: Sin nulos


,visita_id,anuncio_id,fecha,dispositivo
0,100041011,3483,2024-06-27,Desktop
1,100012192,1058,13/04/2024,movil
2,100018389,1581,2024-03-15,desktop


=== contactos.csv | shape=(18005, 6) ===
Tipos: {'contacto_id': dtype('int64'), 'anuncio_id': dtype('int64'), 'inquilino_id': dtype('int64'), 'fecha_contacto': <StringDtype(storage='python', na_value=nan)>, 'canal': <StringDtype(storage='python', na_value=nan)>, 'fecha_respuesta': <StringDtype(storage='python', na_value=nan)>}
Nulos: {'fecha_respuesta': 7413}


,contacto_id,anuncio_id,inquilino_id,fecha_contacto,canal,fecha_respuesta
0,200016520,4323,7789,28/03/2024 17:33,Formulario,NaN
1,200007472,1992,5088,2024-04-04 10:54:48,formulario,2024-04-04 11:06:49
2,200008921,2368,7753,2024-06-16 14:36:26,Formulario,NaN


=== contratos.csv | shape=(1169, 6) ===
Tipos: {'contrato_id': dtype('int64'), 'anuncio_id': dtype('int64'), 'contacto_id': dtype('int64'), 'fecha_firma': <StringDtype(storage='python', na_value=nan)>, 'importe_mensual': <StringDtype(storage='python', na_value=nan)>, 'comision': <StringDtype(storage='python', na_value=nan)>}
Nulos: {'comision': 52}


,contrato_id,anuncio_id,contacto_id,fecha_firma,importe_mensual,comision
0,300000860,3583,200013656,30/06/2024,"1.440,00 €","1.410,66 €"
1,300000122,549,200001987,16/06/2024,"1.170,00 €","1.053,21 €"
2,300000696,2881,200010973,2024-02-02,"899,00 €","919,42 €"


### Análisis de las Tablas de Hechos del Embudo (`visitas`, `contactos`, `contratos`)
1. **`visitas.csv` (`55.661 × 4`):** No presenta valores nulos, pero mezcla formatos de fecha (`YYYY-MM-DD` y `DD/MM/YYYY`) y grafías de `dispositivo` (`Desktop`, `desktop`, `movil`).
2. **`contactos.csv` (`18.005 × 6`):**
   * **El hallazgo forense clave del Día 1:** `fecha_respuesta` tiene **`7.413` valores nulos (`41,17 %` de los registros crudos)**. Lejos de ser un error técnico, este nulo significa que **la agencia inmobiliaria jamás respondió al mensaje del inquilino**.
3. **`contratos.csv` (`1.169 × 6`):**
   * Contiene los `1.169` contratos efectivamente firmados; `importe_mensual` y `comision` vienen como texto con `€`, y existen **`52` contratos con `comision` nula** que debemos conservar para no perder contratos al medir conversión, tratándolos con cuidado en las sumas monetarias.

In [4]:
anuncios = tablas_raw['anuncios']
contactos = tablas_raw['contactos']
visitas = tablas_raw['visitas']
agencias = tablas_raw['agencias']

print(f"1. Grafías distintas de 'zona' en anuncios.csv: {anuncios['zona'].nunique()} (frente a {tablas_raw['zonas']['zona'].nunique()} zonas canónicas)")
print("\n2. Variantes de 'tipo_anuncio' en anuncios.csv (8 grafías):")
print(anuncios['tipo_anuncio'].value_counts())

print("\n3. Variantes de 'ciudad' en anuncios.csv (17 grafías):")
print(anuncios['ciudad'].value_counts().to_dict())

print("\n4. Variantes de 'tipo_anunciante' en anuncios.csv (6 grafías):")
print(anuncios['tipo_anunciante'].value_counts().to_dict())

print("\n5. Variantes de 'canal' en contactos.csv (8 grafías):")
print(contactos['canal'].value_counts().to_dict())

precio_num = (anuncios['precio_mensual'].astype(str)
              .str.replace('€', '', regex=False).str.strip()
              .str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float))
m2_num = (anuncios['m2'].astype(str)
          .str.replace('m2', '', regex=False).str.strip()
          .str.replace('.', '', regex=False).str.replace(',', '.', regex=False).astype(float))

print(f"\n6. Outliers en anuncios -> Precio (<200 € o >8.000 €): {((precio_num < 200) | (precio_num > 8000)).sum()} | m² (<15 o >600): {((m2_num < 15) | (m2_num > 600)).sum()}")
print(f"7. Duplicados por PK -> visitas: {visitas['visita_id'].duplicated().sum()} | contactos: {contactos['contacto_id'].duplicated().sum()}")

1. Grafías distintas de 'zona' en anuncios.csv: 183 (frente a 40 zonas canónicas)

2. Variantes de 'tipo_anuncio' en anuncios.csv (8 grafías):
tipo_anuncio
normal       1614
Normal        802
destacado     761
estandar      478
Destacado     370
std           317
DEST          204
premium       154
Name: count, dtype: int64

3. Variantes de 'ciudad' en anuncios.csv (17 grafías):
{'Sevilla': 621, 'Madrid': 590, 'Valencia': 543, 'Málaga': 463, 'malaga': 239, 'VLC': 237, 'madrid': 233, 'valencia': 231, 'SVQ': 230, 'AGP': 222, 'MAD': 215, 'sevilla': 212, 'VALENCIA': 193, 'MALAGA': 143, 'SEVILLA': 126, 'MADRID': 103, 'Malaga': 99}

4. Variantes de 'tipo_anunciante' en anuncios.csv (6 grafías):
{'agencia': 2060, 'Agencia': 1120, 'profesional': 572, 'particular': 523, 'Particular': 278, 'part': 147}

5. Variantes de 'canal' en contactos.csv (8 grafías):
{'form': 3319, 'formulario': 3309, 'Formulario': 3273, 'Chat': 2256, 'chat': 2246, 'teléfono': 1242, 'telefono': 1215, 'Telefono': 1145}

6. 

### Síntesis Forense del Día 1: Inventario de Minas y Hoja de Ruta para el Día 2

| # | Tabla / Columna | Anomalía Detectada | Impacto si NO se limpia | Regla de Limpieza (Día 2) |
| :-: | :--- | :--- | :--- | :--- |
| **1** | `anuncios.tipo_anuncio` | **8 grafías** (`normal`, `Normal`, `estandar`, `std`, `destacado`, `Destacado`, `DEST`, `premium`) | Rompe cualquier `GROUP BY tipo_anuncio` en 8 filas irreconciliables | Mapear a **`destacado`** (`1.489`) y **`normal`** (`3.211`) |
| **2** | `anuncios.zona` y `ciudad` | **183 grafías de zona** y **17 de ciudad** (`VLC`, `VALENCIA`, `Russafa`, `Cabanyal`...) | Imposibilita cruzar con las **40 zonas canónicas** de `zonas.csv` y las 4 ciudades | Normalizar acentos/alias a las **40 zonas** y **4 ciudades** (`Valencia`, `Madrid`, `Sevilla`, `Málaga`) |
| **3** | `contactos.canal`, `anuncios.tipo_anunciante`, `agencias.plan_suscripcion` | Grafías múltiples (`Formulario`/`form`, `Particular`/`part`, `Basico`/`básico`) | Fragmenta los controles de confusión por canal, tipo de anunciante y plan | Unificar en valores canónicos (`formulario`/`chat`/`telefono`, `agencia`/`particular`, `premium`/`pro`/`basico`/`sin_suscripcion`) |
| **4** | Importes (`€`), `m2` y Fechas | Strings con símbolo `€`, separador de miles `.` y decimal `,`; 3 formatos de fecha | Ordenaciones alfabéticas erróneas y fallo en cálculos de tiempo de respuesta | Convertir importes y `m2` a `float`; parsear fechas distinguiendo ISO (`YYYY-MM-DD`) de formato europeo (`DD/MM/YYYY`) |
| **5** | `visitas` y `contactos` | **386 visitas** y **125 contactos** duplicados por `visita_id` / `contacto_id` | Infla artificialmente el volumen del embudo | Deduplicar por clave primaria (`55.275` visitas y `17.880` contactos únicos) |
| **6** | `contactos.fecha_respuesta` | **7.413 nulos brutos (`7.364` únicos, `41,19 %`)** | Borrar filas con `dropna()` eliminaría el **41,19 %** de los contactos y ocultaría la saturación | Conservar todas las filas; crear `respondido` (`bool`), `horas_respuesta` y `tramo_respuesta` (`<2h`, `2-24h`, `>24h`, `sin_respuesta`) |